<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Data-Structures-and-Algorithms/01-analysis-and-correctness.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Algorithm Analysis and Correctness** {#algorithm-analysis-and-correctness}

Writing code answers only one question: *can a machine execute these instructions?* Algorithm design asks three harder questions: *what result should be produced, why do the steps always produce it, and how does the required work grow as the input becomes larger?*

This chapter develops the language used to answer those questions. The aim is not to memorize a list of complexity classes. It is to learn a repeatable way to move from an informal problem to a precise specification, choose a suitable abstraction, analyze resource usage, and justify correctness before relying on tests.

The central workflow is:

1. state the input assumptions and required output precisely;
2. express the algorithm as finite, unambiguous steps;
3. identify the input-size parameter and count the dominant work;
4. prove that the algorithm terminates and satisfies its postcondition;
5. compare alternatives using both asymptotic cost and practical constraints.


### **Problem Specification and Algorithmic Thinking** {#problem-specification-and-algorithmic-thinking}

A **problem specification** is a contract between the caller and the algorithm. It states what inputs are valid, what output is required, and which conditions must be true when the algorithm finishes. A useful analogy is a navigation request: “take me somewhere nice” is too vague to verify, while “find the shortest driving route from A to B while avoiding toll roads” identifies the input, objective, and constraint.

This precision solves an important problem: two programs can both run without errors while answering different interpretations of the same request. Before selecting a data structure or writing a loop, distinguish four pieces:

- **Input:** the values supplied to the algorithm.
- **Precondition:** assumptions the algorithm is allowed to make, such as “the array is sorted.”
- **Output:** the value or state returned by the algorithm.
- **Postcondition:** the property that must hold after termination.

![A specification connects valid inputs to an algorithm, a required result, and evidence of correctness and efficiency.](assets/algorithm-contract.svg){fig-align="center" width="92%"}

Algorithmic thinking then turns the contract into a controlled sequence of state changes. For example, consider the general task “return the first position of a target in a sequence, or -1 if it is absent.” The invariant behind the scan is simple: before position <code>i</code> is examined, no earlier position contains the target.

**Pseudocode.**

~~~text
FIRST-INDEX(values, target)
    for i from 0 to length(values) - 1
        if values[i] equals target
            return i
    return -1
~~~

The early return is not merely a shortcut. Because positions are inspected from left to right, the first match found is necessarily the smallest valid index. If no step returns, every position has been checked, so <code>-1</code> satisfies the “absent” part of the contract.

<details>
<summary>Python implementation: a specified sequential search</summary>

~~~python
from collections.abc import Sequence
from typing import TypeVar

T = TypeVar("T")


def first_index(values: Sequence[T], target: T) -> int:
    """Return the first target index, or -1 when target is absent."""
    # At the start of each iteration, target is not in values[:index].
    for index, value in enumerate(values):
        if value == target:
            # Scanning left to right makes this the first valid index.
            return index

    # Reaching this line means every position has been ruled out.
    return -1


assert first_index([8, 3, 3, 5], 3) == 1
assert first_index([8, 3, 3, 5], 7) == -1
~~~

</details>

The implementation is preferable to an underspecified “search” function because its behavior for duplicates and missing values is explicit. Its worst-case running time is linear, but correctness is established independently of speed: a faster implementation would still need to satisfy exactly the same contract.

**Practice.** [LeetCode 1 - Two Sum](https://leetcode.com/problems/two-sum/) trains the same specification-first habit: identify exactly what constitutes a valid pair and what must be returned.


### **Abstract Data Types, Interfaces, and Implementations** {#abstract-data-types-interfaces-and-implementations}

An **abstract data type (ADT)** describes a collection by its observable behavior rather than by its physical storage. It is similar to an electrical socket: a device depends on the voltage and plug interface, not on whether the electricity was generated by solar panels, wind, or a turbine. In software, this separation lets client code depend on stable operations while an implementation changes underneath.

Three layers should be kept distinct:

- the **ADT** defines values, operations, and behavioral rules;
- the **interface** gives those operations concrete names and signatures;
- the **implementation** chooses memory layout and algorithms.

A stack ADT, for example, promises **last in, first out** behavior. Its interface may expose <code>push</code>, <code>pop</code>, <code>peek</code>, and <code>is_empty</code>. The same interface can be implemented with a dynamic array or a linked list, each with different memory and performance trade-offs.

![The stack contract is separated from its interface and its possible storage implementations.](assets/adt-layers.svg){fig-align="center" width="90%"}

Internally, an array-backed stack stores the bottom item at index 0 and the current top at the final occupied index. <code>push</code> appends at the top; <code>pop</code> removes only that top item. No caller should reach into the internal list directly, because doing so could violate the stack's ordering rule.


The Stack ADT can be stated as an operation contract. The precondition on <code>pop</code> and <code>peek</code> is part of the abstraction: an implementation must either reject an empty-stack call or define a separate sentinel result.

| ADT operation | Precondition | Required effect or result |
|---|---|---|
| <code>push(x)</code> | None | Add <code>x</code> as the new top and increase size by one. |
| <code>pop()</code> | Stack is not empty | Remove and return the current top. |
| <code>peek()</code> | Stack is not empty | Return the current top without changing the stack. |
| <code>is_empty()</code> | None | Return whether the size is zero. |
| <code>size()</code> | None | Return the number of stored items. |

For <code>n</code> stored items, the representation determines the operation costs:

| Operation | Dynamic-array stack | Linked-list stack |
|---|---:|---:|
| <code>push</code> | amortized $O(1)$; worst $O(n)$ during resize | worst-case $O(1)$ |
| <code>pop</code> | $O(1)$ when removing from the end | $O(1)$ at the head |
| <code>peek</code> | $O(1)$ | $O(1)$ |
| <code>is_empty</code> / <code>size</code> | $O(1)$ | $O(1)$ if size is stored |
| Total storage | $\Theta(n)$ plus unused capacity | $\Theta(n)$ plus one link per node |

The ADT promises LIFO behavior, not a particular complexity. Complexity belongs to the chosen implementation: an array stack may occasionally resize, while a linked stack pays pointer and allocation overhead on every item.

<details>
<summary>Python implementation: an array-backed Stack ADT</summary>

~~~python
from typing import Generic, TypeVar

T = TypeVar("T")


class Stack(Generic[T]):
    """A small LIFO stack whose representation remains private."""

    def __init__(self) -> None:
        # The end of the list is the top of the stack.
        self._items: list[T] = []

    def push(self, item: T) -> None:
        # list.append is amortized O(1).
        self._items.append(item)

    def pop(self) -> T:
        if self.is_empty():
            raise IndexError("pop from an empty stack")
        return self._items.pop()

    def peek(self) -> T:
        if self.is_empty():
            raise IndexError("peek at an empty stack")
        return self._items[-1]

    def is_empty(self) -> bool:
        return len(self._items) == 0

    def __len__(self) -> int:
        return len(self._items)


history = Stack[str]()
history.push("open")
history.push("edit")
assert history.peek() == "edit"
assert history.pop() == "edit"
assert history.pop() == "open"
~~~

</details>

The ADT is “better” than exposing a bare list when the program depends on LIFO behavior: it prevents invalid operations, documents intent, and allows the representation to change. It is not automatically faster; abstraction and performance answer different design questions.

**Practice.** [LeetCode 155 - Min Stack](https://leetcode.com/problems/min-stack/) asks you to preserve a stack interface while enriching its internal representation.


### **Time and Space Complexity** {#time-and-space-complexity}

Complexity describes how resource requirements grow with input size. It is closer to estimating how a journey changes with distance than timing one particular trip: a stopwatch result depends on hardware, language, compiler, and current system load, whereas a growth model explains what happens when the input becomes ten or one thousand times larger.

Let <code>n</code> denote an input-size measure. For an array problem, <code>n</code> is usually the number of elements; for a graph it may be both the number of vertices <code>V</code> and edges <code>E</code>. The choice must be stated because “input size” is not universal.

#### **Time Complexity $T(n)$** {#time-complexity}

Time complexity counts a representative elementary operation, such as comparisons, hash lookups, or edge visits. Suppose duplicate detection compares every pair:

~~~text
HAS-DUPLICATE-BY-PAIRS(values)
    for i from 0 to n - 1
        for j from i + 1 to n - 1
            if values[i] equals values[j]
                return true
    return false
~~~

If no duplicate exists, the number of comparisons is

$$
(n-1) + (n-2) + \cdots + 1
= \frac{n(n-1)}{2}.
$$

Here, <code>n</code> is the number of values, and the fraction counts every unordered pair exactly once. Expanding the expression gives $\frac{1}{2}n^2-\frac{1}{2}n$; for large <code>n</code>, the quadratic term dominates, so the running time is $\Theta(n^2)$.

A set changes the algorithmic strategy. Each value is looked up once, giving expected $\Theta(n)$ time under normal hash-table assumptions.

#### **Space Complexity $S(n)$** {#space-complexity}

Space complexity measures memory that grows with the input. It is useful to distinguish:

- **input space**, occupied by the data supplied by the caller;
- **auxiliary space**, additional memory created by the algorithm;
- **call-stack space**, one frame for each active recursive call.

The pairwise algorithm uses $O(1)$ auxiliary space, while the set-based algorithm may store up to <code>n</code> values and therefore uses $O(n)$ auxiliary space. This is a deliberate **time-space trade-off**: more memory removes repeated comparisons.

<details>
<summary>Python implementation: compare work and memory strategies</summary>

~~~python
from collections.abc import Iterable


def has_duplicate_by_pairs(values: list[int]) -> bool:
    """Use constant auxiliary space but potentially quadratic time."""
    for left in range(len(values)):
        for right in range(left + 1, len(values)):
            if values[left] == values[right]:
                return True
    return False


def has_duplicate_by_set(values: Iterable[int]) -> bool:
    """Use extra memory to obtain expected linear time."""
    seen: set[int] = set()
    for value in values:
        if value in seen:
            return True
        # This state records everything processed so far.
        seen.add(value)
    return False


sample = [4, 8, 1, 8]
assert has_duplicate_by_pairs(sample)
assert has_duplicate_by_set(sample)
~~~

</details>

Neither version is always “better.” The set-based version is usually preferable for large in-memory inputs; pairwise comparison can still be reasonable for tiny inputs or severe memory constraints.

**Practice.** [LeetCode 217 - Contains Duplicate](https://leetcode.com/problems/contains-duplicate/) is a direct exercise in recognizing this time-space trade-off.


### **Best, Average, and Worst Cases** {#best-average-and-worst-cases}

An input size alone does not always determine the exact amount of work. A linear search over <code>n</code> items stops immediately when the target is first, performs several comparisons when the target is near the middle, and examines every item when the target is last or absent.

- The **best case** is the minimum cost among inputs of size <code>n</code>.
- The **worst case** is the maximum cost among inputs of size <code>n</code>.
- The **average case** is an expected cost under a stated probability distribution over inputs.

![Linear search needs one comparison in the best case, several in a typical case, and all n comparisons in the worst case.](assets/linear-search-cases.svg){fig-align="center" width="92%"}

For linear search, best-case time is $\Theta(1)$ and worst-case time is $\Theta(n)$. If the target is equally likely to occupy any of the <code>n</code> positions, a successful search uses

$$
\mathbb{E}[C]
= \frac{1 + 2 + \cdots + n}{n}
= \frac{n+1}{2}
= \Theta(n)
$$

comparisons on average. $C$ is the random variable “number of comparisons,” and $\mathbb{E}[C]$ is its expected value. The exact average is roughly half a scan, but its growth class remains linear.

**Pseudocode.**

~~~text
LINEAR-SEARCH(values, target)
    comparisons <- 0
    for each value in values
        comparisons <- comparisons + 1
        if value equals target
            return found, comparisons
    return not-found, comparisons
~~~

<details>
<summary>Python implementation: make the case-dependent cost visible</summary>

~~~python
def measured_linear_search(values: list[int], target: int) -> tuple[int, int]:
    """Return (index, comparisons) so different cases are observable."""
    comparisons = 0

    for index, value in enumerate(values):
        comparisons += 1
        if value == target:
            return index, comparisons

    return -1, comparisons


values = [7, 2, 9, 4, 1]
assert measured_linear_search(values, 7) == (0, 1)   # best case
assert measured_linear_search(values, 9) == (2, 3)   # middle case
assert measured_linear_search(values, 5) == (-1, 5)  # worst case
~~~

</details>

Worst-case analysis is popular because it provides a guarantee without assuming a friendly input distribution. Average-case claims can be more realistic, but only when their probability assumptions are defensible.

**Practice.** [LeetCode 35 - Search Insert Position](https://leetcode.com/problems/search-insert-position/) shows how a sorted-input precondition improves the worst case from linear search to logarithmic binary search.


### **Asymptotic Notation and Growth Orders** {#asymptotic-notation-and-growth-orders}

Asymptotic notation compares growth after ignoring machine-dependent constants and lower-order terms. This is useful because an algorithm executed on a faster computer can lose its advantage once a worse growth rate meets a sufficiently large input.

- $T(n) \in O(g(n))$ is an **asymptotic upper bound**: beyond some threshold, $T(n)$ grows no faster than a constant multiple of $g(n)$.
- $T(n) \in \Omega(g(n))$ is an **asymptotic lower bound**: beyond some threshold, $T(n)$ grows at least as fast as a constant multiple of $g(n)$.
- $T(n) \in \Theta(g(n))$ is a **tight bound**: both the upper and lower bounds hold.

Formally,

$$
T(n) \in O(g(n))
\iff
\exists c>0,\; \exists n_0\; \text{such that}\;
0 \le T(n) \le c\,g(n)\; \text{for all } n \ge n_0.
$$

The symbol <code>c</code> is a fixed positive multiplier, and <code>n_0</code> is the input size after which the bound must hold. Big-O therefore does **not** mean “exactly equal” and does not by itself mean “worst case”; it states a growth upper bound for whichever case is being analyzed.

![Common growth classes diverge rapidly as input size increases.](assets/complexity-growth.svg){fig-align="center" width="64%"}

*Open visual source: [Wikimedia Commons - Comparison computational complexity](https://commons.wikimedia.org/wiki/File:Comparison_computational_complexity.svg).*

Typical orders, from slower to faster growth, are:

| Order | Informal behavior | Common source |
|---|---|---|
| $O(1)$ | work does not grow with <code>n</code> | array index, hash lookup on average |
| $O(\log n)$ | repeatedly discard a constant fraction | binary search, balanced-tree lookup |
| $O(n)$ | inspect each item a constant number of times | scan, prefix construction |
| $O(n\log n)$ | logarithmic levels with linear work per level | efficient comparison sorting |
| $O(n^2)$ | inspect many pairs | nested all-pairs loops |
| $O(2^n)$ | branch on many binary choices | naive subset search |
| $O(n!)$ | enumerate permutations | brute-force ordering |

<details>
<summary>Python implementation: compare growth estimates numerically</summary>

~~~python
import math


def growth_estimates(n: int) -> dict[str, float]:
    """Return representative operation counts for common growth classes."""
    if n <= 0:
        raise ValueError("n must be positive")

    return {
        "constant": 1,
        "logarithmic": math.log2(n),
        "linear": n,
        "linearithmic": n * math.log2(n),
        "quadratic": n * n,
    }


for size in (10, 1_000, 1_000_000):
    costs = growth_estimates(size)
    print(size, {name: round(value, 2) for name, value in costs.items()})
~~~

</details>

Asymptotic order is a filter, not a complete benchmark. Constants, cache behavior, input distribution, and implementation overhead still matter after unsuitable growth rates have been ruled out.

**Practice.** [LeetCode 121 - Best Time to Buy and Sell Stock](https://leetcode.com/problems/best-time-to-buy-and-sell-stock/) rewards replacing a quadratic comparison of all day pairs with a single linear scan.


### **Loop and Recursion Analysis** {#loop-and-recursion-analysis}

Loops and recursion both express repetition. A loop keeps its state in variables and returns control to a condition; recursion keeps one state per active call and reduces a problem to smaller instances of itself. The better form is the one that makes progress, termination, and cost easiest to reason about.

#### **Loop-Based Analysis** {#loop-based-analysis}

For a loop, identify how often its control variable changes and how much work each iteration performs. A loop that increments <code>i</code> from 0 to <code>n - 1</code> has <code>n</code> iterations. A loop that doubles <code>i</code> until it reaches <code>n</code> has approximately $\log_2 n$ iterations because after <code>k</code> steps, $i=2^k$.

~~~text
i <- 1
while i < n
    perform constant work
    i <- 2 * i
~~~

![A while loop repeatedly checks its condition, executes the body, and updates state before checking again.](assets/while-loop-flow.svg){fig-align="center" width="50%"}

*Open visual source: [Wikimedia Commons - While-loop diagram](https://commons.wikimedia.org/wiki/File:While-loop-diagram.svg).*

Nested loops are not automatically quadratic. Their bounds matter: if the inner pointer never moves backward across the whole execution, two visually nested loops can still perform only linear total work.

#### **Recursive Algorithms and Recurrence Relations** {#recursive-algorithms-and-recurrence-relations}

A recursive algorithm needs a **base case** that stops and a **recursive case** that moves closer to it. Its running time is often expressed by a recurrence. Naive Fibonacci obeys

$$
T(n)=T(n-1)+T(n-2)+\Theta(1).
$$

The two recursive terms represent the two child calls; $\Theta(1)$ represents the addition and control work in the current call. The call tree contains overlapping subproblems, so the total number of calls grows exponentially.

![Naive Fibonacci recursion branches into overlapping calls such as fib(3) and fib(2).](assets/recursion-unfolding.svg){fig-align="center" width="88%"}

**Pseudocode with memoization.**

~~~text
FIBONACCI(n, memo)
    if n <= 1
        return n
    if n is in memo
        return memo[n]
    memo[n] <- FIBONACCI(n - 1, memo) + FIBONACCI(n - 2, memo)
    return memo[n]
~~~

Memoization ensures each integer from 0 through <code>n</code> is solved once. The time becomes $\Theta(n)$ and the memo plus call stack use $\Theta(n)$ space.

<details>
<summary>Python implementation: iterative and memoized recursive forms</summary>

~~~python
def fibonacci_iterative(n: int) -> int:
    """Compute F(n) with linear time and constant auxiliary space."""
    if n < 0:
        raise ValueError("n must be non-negative")

    previous, current = 0, 1
    for _ in range(n):
        # After k iterations: previous = F(k), current = F(k + 1).
        previous, current = current, previous + current
    return previous


def fibonacci_memoized(n: int, memo: dict[int, int] | None = None) -> int:
    """Compute F(n) recursively without recomputing subproblems."""
    if n < 0:
        raise ValueError("n must be non-negative")
    if n <= 1:
        return n

    if memo is None:
        memo = {}
    if n not in memo:
        memo[n] = (
            fibonacci_memoized(n - 1, memo)
            + fibonacci_memoized(n - 2, memo)
        )
    return memo[n]


assert fibonacci_iterative(10) == 55
assert fibonacci_memoized(10) == 55
~~~

</details>

Iteration avoids call-stack overhead and is often preferable for a simple linear state transition. Recursion is clearer for naturally recursive structures such as trees, divide-and-conquer algorithms, and backtracking.

**Practice.** [LeetCode 70 - Climbing Stairs](https://leetcode.com/problems/climbing-stairs/) exposes the same overlapping-subproblem recurrence as Fibonacci.


### **Correctness Proofs: Induction, Contradiction, and Loop Invariants** {#correctness-proofs}

Testing demonstrates that an algorithm works on selected inputs; a correctness proof explains why it works for **every** input satisfying the precondition. The distinction resembles quality inspection versus an engineering argument: checking several bridges is useful evidence, but it does not replace reasoning about loads, materials, and all allowed conditions.

Three proof techniques appear repeatedly:

- **Mathematical induction** proves a base case and then proves that truth for size <code>k</code> implies truth for size <code>k + 1</code>. It naturally matches recursive algorithms.
- **Contradiction** assumes the desired claim is false and derives an impossibility. It is useful for greedy choices and lower-bound arguments.
- A **loop invariant** is a property that remains true before and after every iteration. It connects local updates to the final postcondition.

![A loop-invariant proof has initialization, maintenance, and termination stages.](assets/loop-invariant.svg){fig-align="center" width="90%"}

Consider an algorithm that returns the maximum item of a non-empty sequence.

**Pseudocode.**

~~~text
MAXIMUM(values)
    require length(values) > 0
    best <- values[0]
    for i from 1 to length(values) - 1
        if values[i] > best
            best <- values[i]
    return best
~~~

Use the invariant: **before iteration <code>i</code>, <code>best</code> is the maximum of <code>values[0:i]</code>.**

1. **Initialization:** before <code>i = 1</code>, the processed prefix contains only <code>values[0]</code>, so the invariant holds.
2. **Maintenance:** comparing <code>values[i]</code> with the old prefix maximum makes <code>best</code> the maximum of the prefix extended through <code>i</code>.
3. **Termination:** when the loop ends, the processed prefix is the entire sequence, so <code>best</code> is the required global maximum.

Termination also needs justification: <code>i</code> increases by one and is bounded above by the finite length of the sequence.

<details>
<summary>Python implementation: code aligned with the invariant</summary>

~~~python
from collections.abc import Sequence
from typing import TypeVar

T = TypeVar("T")


def maximum(values: Sequence[T]) -> T:
    """Return the greatest item in a non-empty comparable sequence."""
    if not values:
        raise ValueError("maximum requires a non-empty sequence")

    # Invariant before index i:
    # best is the maximum value in values[:i].
    best = values[0]
    for index in range(1, len(values)):
        if values[index] > best:
            best = values[index]

    # At termination, values[:len(values)] is the whole input.
    return best


assert maximum([-4, 7, 2, 7, 1]) == 7
~~~

</details>

A good implementation often mirrors its proof: variables have clear meanings, each update preserves an invariant, and the exit condition turns that invariant into the postcondition.

**Practice.** [LeetCode 283 - Move Zeroes](https://leetcode.com/problems/move-zeroes/) is well suited to a loop-invariant proof over processed and unprocessed array regions.


### **Amortized Analysis** {#amortized-analysis}

**Amortized analysis** bounds the average cost per operation over a worst-case sequence of operations, without assuming a probability distribution. Think of paying a small amount into a maintenance fund every month: most months are cheap, while the accumulated credit covers an occasional expensive repair.

A dynamic array illustrates the idea. It stores elements in a fixed-capacity contiguous block. Appending is cheap while unused capacity remains. When the block is full, the implementation allocates a block of double capacity, copies existing elements, and then appends the new item.

![A dynamic array occasionally allocates a larger block and copies its occupied elements.](assets/dynamic-array-growth.svg){fig-align="center" width="78%"}

*Open visual source: [Wikimedia Commons - Dynamic array](https://commons.wikimedia.org/wiki/File:Dynamic_array.svg).*

**Pseudocode.**

~~~text
APPEND(value)
    if size equals capacity
        new_storage <- array with 2 * capacity slots
        copy all size elements into new_storage
        storage <- new_storage
        capacity <- 2 * capacity
    storage[size] <- value
    size <- size + 1
~~~


As a sequence-like ADT, a dynamic array normally exposes more than <code>append</code>:

| ADT operation | Required behavior | Dynamic-array cost |
|---|---|---:|
| <code>get(i)</code> | Return the item at valid index <code>i</code>. | $O(1)$ |
| <code>set(i, x)</code> | Replace the item at valid index <code>i</code>. | $O(1)$ |
| <code>append(x)</code> | Add <code>x</code> after the current last item. | amortized $O(1)$; worst $O(n)$ |
| <code>pop_back()</code> | Remove and return the last item. | $O(1)$ |
| <code>insert(i, x)</code> | Insert at index <code>i</code> and preserve order. | $O(n)$ |
| <code>delete(i)</code> | Remove index <code>i</code> and close the gap. | $O(n)$ |

The two constant-time indexed operations follow from contiguous storage. Middle insertion and deletion are linear because a suffix may have to shift. The table separates the ADT's required behavior from the resizing strategy used to implement it.

A single resize may cost $\Theta(n)$, but resizes do not happen on every append. With capacities $1,2,4,8,\ldots$, the total number of copied items before reaching size <code>n</code> is

$$
1+2+4+\cdots+2^k < 2n.
$$

Adding the <code>n</code> ordinary writes gives fewer than <code>3n</code> basic moves across <code>n</code> appends. Dividing total cost by the number of operations yields $O(1)$ **amortized** time per append, even though an individual append can still be linear.

<details>
<summary>Python implementation: expose capacity growth explicitly</summary>

~~~python
from typing import Generic, TypeVar, cast

T = TypeVar("T")


class DynamicArray(Generic[T]):
    """A minimal dynamic array that doubles when full."""

    def __init__(self) -> None:
        self._capacity = 1
        self._size = 0
        self._storage: list[T | None] = [None] * self._capacity

    def _check_index(self, index: int) -> None:
        if index < 0 or index >= self._size:
            raise IndexError("dynamic-array index out of range")

    def get(self, index: int) -> T:
        self._check_index(index)
        return cast(T, self._storage[index])

    def set(self, index: int, value: T) -> None:
        self._check_index(index)
        self._storage[index] = value

    def append(self, value: T) -> None:
        if self._size == self._capacity:
            self._resize(self._capacity * 2)

        self._storage[self._size] = value
        self._size += 1

    def pop_back(self) -> T:
        if self._size == 0:
            raise IndexError("pop from an empty dynamic array")

        self._size -= 1
        value = cast(T, self._storage[self._size])
        self._storage[self._size] = None
        return value

    def _resize(self, new_capacity: int) -> None:
        new_storage: list[T | None] = [None] * new_capacity

        # This copy is expensive, but capacity doubling makes it infrequent.
        for index in range(self._size):
            new_storage[index] = self._storage[index]

        self._storage = new_storage
        self._capacity = new_capacity

    def __len__(self) -> int:
        return self._size

    @property
    def capacity(self) -> int:
        return self._capacity


values = DynamicArray[int]()
for number in range(9):
    values.append(number)

assert len(values) == 9
assert values.capacity == 16
assert values.get(3) == 3
values.set(3, 30)
assert values.get(3) == 30
assert values.pop_back() == 8
assert len(values) == 8
~~~

</details>

Amortized cost is stronger than saying an operation is “usually fast”: it guarantees that **every sufficiently long operation sequence** has bounded average cost. It should not be confused with average-case analysis, which depends on a probability model.

**Practice.** [LeetCode 739 - Daily Temperatures](https://leetcode.com/problems/daily-temperatures/) uses the same style of reasoning: each index enters and leaves a monotonic stack at most once.


### **Practical Complexity Selection** {#practical-complexity-selection}

Choosing an algorithm is a constrained engineering decision, not a contest to obtain the smallest Big-O symbol in isolation. Start from correctness, then compare the expected input scale, worst-case guarantees, memory budget, update frequency, ordering requirements, implementation risk, and the cost of maintaining the code.

| Question | Why it changes the choice |
|---|---|
| Is the input already sorted? | Binary search and two-pointer methods may become available. |
| Are there many queries over mostly static data? | Preprocessing such as sorting or prefix sums can pay for itself. |
| Is memory constrained? | An in-place $O(n^2)$ method may beat an $O(n)$ method requiring a large table. |
| Is latency predictable? | Worst-case guarantees may matter more than expected performance. |
| Is <code>n</code> always small? | A simple quadratic method can be clearer and fast enough. |
| Does the representation fit the access pattern? | Arrays favor indexing; hash maps favor exact-key lookup; trees preserve order. |

A disciplined selection process is:

1. eliminate candidates that do not satisfy the specification;
2. estimate realistic input sizes and operation frequencies;
3. compare time and auxiliary-space growth;
4. account for assumptions such as sortedness or expected constant-time hashing;
5. benchmark representative inputs only after the theoretical model is understood;
6. document the trade-off so future changes can be evaluated consistently.

The main lesson of this chapter is that analysis and proof support each other. A precise invariant often reveals the correct implementation, while a careful cost model exposes which state or repeated work should be redesigned.

**Practice.** [LeetCode 15 - 3Sum](https://leetcode.com/problems/3sum/) is a useful final exercise in specification, duplicate handling, asymptotic improvement, and selecting a structure-aware strategy.
